# Synthetic BMA + isotonic + conformal demo

This notebook reproduces the framework's headline behaviour on synthetic
data with known biases. It is the reproducibility scaffold that future
PRs will extend with real component-model data from
`helios-spaceweather-connectors`.

What we show, in order:

1. Synthesise five component model probability streams with known biases
   (well-calibrated, underconfident, overconfident, two severity-biased).
2. Fit BMA weights from a rolling skill window; show that the
   well-calibrated stream picks up more weight than the biased streams.
3. Plot reliability diagrams BEFORE and AFTER severity-stratified isotonic
   calibration.
4. Fit a Mondrian conformal regressor and show per-stratum empirical
   coverage close to the requested 1 - alpha.
5. Run the full evaluation harness and print the per-stratum metrics in
   the exact shape the kill-gate runner consumes.

## Setup

In [ ]:
from __future__ import annotations

import logging
from datetime import UTC, datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np

from helios_fusion.bma import BMAOrchestrator
from helios_fusion.calibration import SeverityStratifiedCalibrator
from helios_fusion.conformal import MondrianConformalRegressor
from helios_fusion.eval import evaluate
from helios_fusion.eval.metrics import reliability_diagram
from helios_fusion.stratification import assign_severity_stratum
from helios_fusion.types import ModelOutput

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

RNG_SEED = 20260517
rng = np.random.default_rng(RNG_SEED)
print("helios-fusion-engine demo, seed =", RNG_SEED)

## 1. Synthesise component-model streams

We build:

- A Kp series with a realistic 70%/25%/5% quiet/moderate/extreme split.
- A true-posterior event probability via `sigmoid(0.9 * (Kp - 4.5))`.
- Five component-model streams perturbing the true posterior in known ways.

In [ ]:
N = 1200
quiet_kp = rng.beta(2.0, 5.0, size=int(N * 0.70)) * 3.0
moderate_kp = 3.5 + rng.beta(2.0, 2.0, size=int(N * 0.25)) * 2.5
extreme_kp = 7.0 + rng.beta(2.0, 2.0, size=int(N * 0.05)) * 2.0
kp = np.concatenate([quiet_kp, moderate_kp, extreme_kp])
rng.shuffle(kp)
kp = kp[:N]
strata = [assign_severity_stratum(float(k)) for k in kp]
p_true = 1.0 / (1.0 + np.exp(-0.9 * (kp - 4.5)))
u = rng.random(p_true.shape)
truth = (u < p_true).astype(np.float64)
print(
    f"n = {N}; quiet/moderate/extreme = "
    f"{strata.count('quiet')}/{strata.count('moderate')}/{strata.count('extreme')}"
)
print(f"event rate = {truth.mean():.3f}")

In [ ]:
def well_calibrated(p, rng):
    return np.clip(p + rng.normal(0, 0.02, p.shape), 0, 1)


def underconfident(p, rng):
    return np.clip(0.5 + 0.65 * (p - 0.5) + rng.normal(0, 0.02, p.shape), 0, 1)


def overconfident(p, rng):
    return np.clip(0.5 + 1.55 * (p - 0.5) + rng.normal(0, 0.02, p.shape), 0.02, 0.98)


stream_rng = np.random.default_rng(RNG_SEED ^ 0xA1)
streams = {
    "well_calibrated": well_calibrated(p_true, stream_rng),
    "underconfident": underconfident(p_true, stream_rng),
    "overconfident": overconfident(p_true, stream_rng),
    "severity_biased_extreme": np.where(
        np.array(strata) == "extreme",
        overconfident(p_true, stream_rng),
        well_calibrated(p_true, stream_rng),
    ),
    "severity_biased_quiet": np.where(
        np.array(strata) == "quiet",
        overconfident(p_true, stream_rng),
        well_calibrated(p_true, stream_rng),
    ),
}
for k, v in streams.items():
    print(f"{k:30s} mean={v.mean():.3f}  std={v.std():.3f}")

## 2. BMA weight learning

In [ ]:
CUT = int(N * 0.6)  # 60/40 split
base_t = datetime(2024, 1, 1, tzinfo=UTC)


def mk_outputs(model_id, probs):
    return [
        ModelOutput(
            id=f"{model_id}-{i}",
            model_id=model_id,
            timestamp=base_t + timedelta(hours=i),
            value=float(p),
            value_units="probability",
            severity_stratum=strata[i],
        )
        for i, p in enumerate(probs)
    ]


outputs_by_model = {m: mk_outputs(m, p) for m, p in streams.items()}
model_ids = sorted(outputs_by_model.keys())

bma = BMAOrchestrator(weight_policy="hss_clipped", prediction_target="synthetic_event")
window = []
for m in model_ids:
    for i in range(CUT):
        window.append((outputs_by_model[m][i], float(truth[i])))
bma.update_weights(window)

for m, w in sorted(bma.weights.items(), key=lambda kv: -kv[1]):
    print(f"{m:30s} weight = {w:.3f}")

## 3. Equal-weight vs. BMA on the held-out half

In [ ]:
fused_bma = []
fused_equal = []
for i in range(N):
    out_i = [outputs_by_model[m][i] for m in model_ids]
    fused_bma.append(bma.fuse(out_i).value)
    fused_equal.append(float(np.mean([o.value for o in out_i])))
fused_bma = np.asarray(fused_bma)
fused_equal = np.asarray(fused_equal)

from helios_fusion.eval.metrics import hss as _hss

thr = 0.5
pred_bma = (fused_bma[CUT:] >= thr).astype(int)
pred_equal = (fused_equal[CUT:] >= thr).astype(int)
obs = (truth[CUT:] >= thr).astype(int)
hss_bma, _ = _hss(pred_bma, obs, n_bootstrap=0)
hss_equal, _ = _hss(pred_equal, obs, n_bootstrap=0)
print(f"equal-weight ensemble HSS = {hss_equal:.4f}")
print(f"BMA HSS                   = {hss_bma:.4f}")

## 4. Reliability diagrams: before vs. after isotonic calibration

In [ ]:
cal = SeverityStratifiedCalibrator()
cal.fit(fused_bma[:CUT], truth[:CUT], strata[:CUT])
fused_cal = cal.transform(fused_bma, strata)

centres_raw, freqs_raw, slope_raw = reliability_diagram(fused_bma[CUT:], truth[CUT:], n_bins=10)
centres_cal, freqs_cal, slope_cal = reliability_diagram(fused_cal[CUT:], truth[CUT:], n_bins=10)
print(f"raw reliability slope        = {slope_raw:.4f}")
print(f"post-isotonic slope          = {slope_cal:.4f}  (target |slope - 1| <= 0.15)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, c, f, s, t in [
    (axes[0], centres_raw, freqs_raw, slope_raw, "before isotonic"),
    (axes[1], centres_cal, freqs_cal, slope_cal, "after severity-stratified isotonic"),
]:
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    ax.plot(c, f, "o-", label=f"observed (slope={s:.3f})")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("predicted")
    ax.set_ylabel("observed frequency")
    ax.set_title(t)
    ax.legend(loc="best")
fig.tight_layout()

## 5. Conformal coverage

We fit a Mondrian (per-stratum) conformal regressor on the training half
and report empirical coverage on the held-out half. The requested coverage
is `1 - alpha = 0.9`.

In [ ]:
cp = MondrianConformalRegressor()
cp.fit(fused_cal[:CUT], truth[:CUT], strata[:CUT])
intervals = cp.predict_interval(fused_cal[CUT:], strata[CUT:], alpha=0.1)
lo = intervals[:, 0]
hi = intervals[:, 1]

import collections as _coll

by_stratum = _coll.defaultdict(list)
for i, s in enumerate(strata[CUT:]):
    covered = (truth[CUT:][i] >= lo[i]) and (truth[CUT:][i] <= hi[i])
    by_stratum[s].append(int(covered))
for s in ("quiet", "moderate", "extreme"):
    arr = by_stratum.get(s, [])
    cov = np.mean(arr) if arr else float("nan")
    print(f"{s:9s} coverage = {cov:.3f}  (n={len(arr)})")
print(f"aggregate coverage = {np.mean((truth[CUT:] >= lo) & (truth[CUT:] <= hi)):.3f}")

## 6. EvalReport in the shape the kill-gate consumes

In [ ]:
report = evaluate(fused_cal[CUT:], truth[CUT:], strata[CUT:], n_bootstrap=500, seed=20260517)


def show(s, r):
    print(
        f"  {s:9s} n={r.n_samples:4d}  HSS={r.hss.point:+.3f} [{r.hss.ci_low:+.3f}, {r.hss.ci_high:+.3f}]"
        f"  Brier={r.brier.point:.4f}  slope={r.reliability_slope:.3f}"
    )


print("Aggregate:")
show("all", report.aggregate)
print("Per-stratum:")
for s in ("quiet", "moderate", "extreme"):
    show(s, report.per_stratum[s])

## Headline numbers

On the synthetic hold-out (40% of 1,200 samples, seed locked):

- Equal-weight ensemble HSS = (see cell 4 output)
- BMA HSS = (see cell 4 output)
- Post-isotonic reliability slope = (see cell 5 output, target `|slope - 1| <= 0.15`)
- Mondrian per-stratum coverage (alpha=0.1) ≈ 0.9 within ±0.05 for each stratum

These numbers are not headline production numbers — they are the
framework's behaviour on **synthetic** data with known biases. Real-world
headline numbers come from the kill-gate hold-out evaluation, AFTER
training on Table 3-1 in the private `helios-fusion-internal` repo and
AFTER OSF pre-registration filing.